In [ ]:
# =========================================================
# 11_BRONZE_CUSTOMERS
# Simple incremental load from Landing to Bronze
# No audit
# No watermark
# =========================================================

# ---------------------------------------------------------
# 1. CREATE BRONZE TABLE
# ---------------------------------------------------------

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.bronze.customers (
        customer_id STRING,
        first_name STRING,
        last_name STRING,
        email STRING,
        country STRING,
        city STRING,
        customer_type STRING,
        registration_date STRING,
        last_updated STRING,
        event_id STRING,
        batch_id STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
""")

# ---------------------------------------------------------
# 2. INSERT ONLY NEW EVENTS
# ---------------------------------------------------------

spark.sql("""
    INSERT INTO workspace.bronze.customers
    SELECT
        l.customer_id,
        l.first_name,
        l.last_name,
        l.email,
        l.country,
        l.city,
        l.customer_type,
        l.registration_date,
        l.last_updated,
        l.event_id,
        l.batch_id,
        l.load_timestamp
    FROM workspace.landing.customers l
    WHERE NOT EXISTS (
        SELECT 1
        FROM workspace.bronze.customers b
        WHERE b.batch_id = l.batch_id
          AND b.event_id = l.event_id
    )
""")

# ---------------------------------------------------------
# 3. SHOW RESULT
# ---------------------------------------------------------

result = spark.sql("""
    SELECT
        batch_id,
        COUNT(*) AS rows_in_bronze
    FROM workspace.bronze.customers
    GROUP BY batch_id
    ORDER BY batch_id
""")

display(result)